In [ ]:
from __future__ import annotations

import pickle

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles
from scipy.stats import ttest_ind, spearmanr
from scipy.optimize import curve_fit
def exp_decay(x, a, tau, b):
    return a * np.exp(-x / tau) + b

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "07.MF1.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

print(f"Resolved ROI target: {target}")
print(f"Using top-k = {top_k}")
print(f"Raster shape after binning {raster_4d.shape}")

# shape (units, time, images)
raster_3d = np.nanmean(raster_4d, axis=3)
scores = tut.rank_images_by_response(raster_3d)

idx_topk = scores[:top_k]
trial_3d = raster_3d[:, :, idx_topk]
print(f"Trial averaged, all data {raster_3d.shape}")
print(f"Trial averaged, top-k shape {trial_3d.shape}")

In [ ]:
rr, rrdv = tut.tuning_rdm(raster_3d, scores)

In [ ]:
fig,ax = plt.subplots(1,1)
sns.heatmap(1-rr2, ax=ax, square=True)
plt.show()

fig,ax = plt.subplots(1,1)
sns.heatmap(1-rr, ax=ax, square=True)
plt.show()

In [ ]:
def lag_model(M):
    n = M.shape[0]
    M_hat = np.zeros_like(M)

    for k in range(n):
        diag = np.diag(M, k=k)
        mean = diag.mean()

        M_hat += np.diag(np.full(len(diag), mean), k=k)
        if k > 0:
            M_hat += np.diag(np.full(len(diag), mean), k=-k)

    return M_hat

M_hat = lag_model(rr2)

def lag_fit_score(M, M_hat):
    # flatten
    x = M.flatten()
    y = M_hat.flatten()

    # correlation (simple and interpretable)
    r = np.corrcoef(x, y)[0,1]

    # R^2
    r2 = r**2

    return r, r2

r, r2 = lag_fit_score(rr, M_hat)
print(r, r2)

# def residual_ratio(M, M_hat):
#     resid = M - M_hat
#     return np.var(resid) / np.var(M)


In [ ]:
# rr2, rrdv2 = tut.tuning_rdm(raster_3d, idx_topk)
rr2, rrdv2 = tut.tuning_rdm(raster_3d, np.random.choice(scores, 30))
T = rrdv2.shape[0]

# normalize each timepoint (important!)
rdv_z = (rrdv2 - rrdv2.mean(axis=1, keepdims=True)) / (rrdv2.std(axis=1, keepdims=True) + 1e-8)

max_lag = 100  # adjust if needed
lags = np.arange(1, max_lag)

lag_sim = []

for d in lags:
    sims = []
    for t in range(T - d):
        s = np.dot(rdv_z[t], rdv_z[t + d]) / rdv_z.shape[1]  # cosine / corr equivalent after z-score
        sims.append(s)
    lag_sim.append(np.mean(sims))

lag_sim = np.array(lag_sim)

plt.figure(figsize=(4,3))
plt.plot(lags, lag_sim, linewidth=2)
plt.xlabel('Time lag')
plt.ylabel('Geometry similarity')
plt.title('Lag-dependent geometry similarity')
plt.axhline(0, linestyle='--', color='gray')
plt.tight_layout()
plt.show()

rho, pval = spearmanr(lags, lag_sim)
print(f"Spearman rho (lag vs similarity): {rho:.3f}, p = {pval:.3e}")

popt, _ = curve_fit(exp_decay, lags, lag_sim, maxfev=10000)
fit = exp_decay(lags, *popt)

residual = np.mean((lag_sim - fit)**2)
print("Deviation from exponential:", residual)

slope = np.abs(np.diff(lag_sim))
flatness = np.mean(slope < 0.01)  # threshold tunable
print("Fraction flat:", flatness)

In [ ]:
T = rrdv.shape[0]

# normalize each timepoint (important!)
rdv_z = (rrdv - rrdv.mean(axis=1, keepdims=True)) / (rrdv.std(axis=1, keepdims=True) + 1e-8)

max_lag = 100  # adjust if needed
lags = np.arange(1, max_lag)

lag_sim = []

for d in lags:
    sims = []
    for t in range(T - d):
        s = np.dot(rdv_z[t], rdv_z[t + d]) / rdv_z.shape[1]  # cosine / corr equivalent after z-score
        sims.append(s)
    lag_sim.append(np.mean(sims))

lag_sim = np.array(lag_sim)

plt.figure(figsize=(4,3))
plt.plot(lags, lag_sim, linewidth=2)
plt.xlabel('Time lag')
plt.ylabel('Geometry similarity')
plt.title('Lag-dependent geometry similarity')
plt.axhline(0, linestyle='--', color='gray')
plt.tight_layout()
plt.show()

rho, pval = spearmanr(lags, lag_sim)
print(f"Spearman rho (lag vs similarity): {rho:.3f}, p = {pval:.3e}")

popt, _ = curve_fit(exp_decay, lags, lag_sim, maxfev=10000)
fit = exp_decay(lags, *popt)

residual = np.mean((lag_sim - fit)**2)
print("Deviation from exponential:", residual)

slope = np.abs(np.diff(lag_sim))
flatness = np.mean(slope < 0.01)  # threshold tunable
print("Fraction flat:", flatness)